Importando Bibliotecas.

In [18]:
import zipfile
from pathlib import Path
import shutil

Classe de Controle

In [19]:
                
class Extract:
    def __init__(self, path_origem, path_staging):
        self.path_origem = Path(path_origem)
        self.path_staging = Path(path_staging)

        self.total_zip = 0
        self.total_csv = 0
        self.total_erros = 0

        self.arquivos_extraidos = []
        self.arquivos_erro = []

    def criar_diretorio(self):
        self.path_staging.mkdir(
            parents=True,
            exist_ok=True
        )
     
    def localizar_zip(self):
        arquivos_zip = list(
            self.path_origem.rglob("*.zip")
        )

        self.total_zip = len(arquivos_zip)

        print("=" * 70)
        print("ETAPA 1 - LOCALIZAÇÃO DOS ARQUIVOS ZIP")
        print("=" * 70)

        print(
            f"Total de arquivos ZIP encontrados: "
            f"{self.total_zip}"
        )

        for arquivo_zip in arquivos_zip:
            print(f"ZIP | {arquivo_zip}")

        return arquivos_zip
    
    def extrair_csv(self, arquivo_zip):
        nome_pasta = arquivo_zip.parent.name
        nome_zip = arquivo_zip.stem

        print(f"\nPROCESSANDO: {arquivo_zip}")

        with zipfile.ZipFile(arquivo_zip, "r") as zip_ref:
            arquivos_csv = [
                nome
                for nome in zip_ref.namelist()
                if nome.lower().endswith(".csv")
                and not nome.endswith("/")
            ]

            print(f"CSV encontrados: {len(arquivos_csv)}")

            for nome_csv in arquivos_csv:
                try:
                    nome_csv_original = Path(nome_csv).name
                    nome_csv_base = Path(nome_csv).stem

                    nome_saida = (
                        f"{nome_pasta}_"
                        f"{nome_zip}_"
                        f"{nome_csv_base}.csv"
                    )

                    arquivo_saida = (
                        self.path_staging / nome_saida
                    )

                    with zip_ref.open(nome_csv) as origem:
                        with open(arquivo_saida, "wb") as destino:
                            shutil.copyfileobj(origem, destino)

                    self.total_csv += 1

                    self.arquivos_extraidos.append({
                        "pasta": nome_pasta,
                        "zip": arquivo_zip.name,
                        "csv_original": nome_csv_original,
                        "arquivo_saida": nome_saida
                    })

                    print(
                        f"   OK | {nome_csv_original} "
                        f"-> {nome_saida}"
                    )

                except Exception as erro:
                    self.registrar_erro(
                        tipo="CSV",
                        arquivo_zip=arquivo_zip,
                        csv=nome_csv,
                        erro=erro
                    )

    def registrar_erro(self, tipo, arquivo_zip, csv, erro):
        self.total_erros += 1

        self.arquivos_erro.append({
            "tipo": tipo,
            "zip": str(arquivo_zip),
            "csv": csv,
            "erro": str(erro)
        })

        print(
            f"ERRO {tipo} | {arquivo_zip} | "
            f"{csv} | {erro}"
        )

    def executar(self):
        self.criar_diretorio()

        arquivos_zip = self.localizar_zip()

        print("\n")
        print("=" * 70)
        print("ETAPA 2 - EXTRAÇÃO DOS CSV")
        print("=" * 70)

        for arquivo_zip in arquivos_zip:
            try:
                self.extrair_csv(arquivo_zip)

            except Exception as erro:
                self.registrar_erro(
                    tipo="ZIP",
                    arquivo_zip=arquivo_zip,
                    csv="",
                    erro=erro
                )

        self.relatorio_final()
        
    def relatorio_final(self):
        print("\n")
        print("=" * 70)
        print("RELATÓRIO FINAL")
        print("=" * 70)

        print(f"ZIP encontrados : {self.total_zip}")
        print(f"CSV extraídos   : {self.total_csv}")
        print(f"Erros           : {self.total_erros}")
        print(f"Pasta de saída  : {self.path_staging}")

        print("\n")
        print("=" * 70)
        print("ARQUIVOS CSV NA PASTA DE SAÍDA")
        print("=" * 70)

        arquivos_saida = list(
            self.path_staging.glob("*.csv")
        )

        for arquivo in arquivos_saida:
            print(f"CSV | {arquivo.name}")

        print(
            f"\nTotal de CSV na pasta: {len(arquivos_saida)}"
        )

        if self.arquivos_erro:
            print("\n")
            print("=" * 70)
            print("ERROS")
            print("=" * 70)

            for erro in self.arquivos_erro:
                print(
                    f"Tipo: {erro['tipo']}\n"
                    f"ZIP: {erro['zip']}\n"
                    f"CSV: {erro['csv']}\n"
                    f"Erro: {erro['erro']}\n"
                )
        else:
            print("\nNenhum erro encontrado.")
                                                                                                                                                                                                                                                                                                                                  

Processamento Principal.

In [20]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================
pasta_origem = Path(r"dados\raw\cliente\zip")
pasta_staging = Path(r"dados\raw\cliente\csv")

# ============================================================
# EXECUÇÃO
# ============================================================
extract = Extract(
    path_origem=pasta_origem,
    path_staging=pasta_staging
)

extract.executar()

ETAPA 1 - LOCALIZAÇÃO DOS ARQUIVOS ZIP
Total de arquivos ZIP encontrados: 4
ZIP | dados\raw\cliente\zip\CLIENTE1\relatorio1.zip
ZIP | dados\raw\cliente\zip\CLIENTE1\relatorio2.zip
ZIP | dados\raw\cliente\zip\CLIENTE2\relatorio1.zip
ZIP | dados\raw\cliente\zip\CLIENTE2\relatorio2.zip


ETAPA 2 - EXTRAÇÃO DOS CSV

PROCESSANDO: dados\raw\cliente\zip\CLIENTE1\relatorio1.zip
CSV encontrados: 1
   OK | dados.csv -> CLIENTE1_relatorio1_dados.csv

PROCESSANDO: dados\raw\cliente\zip\CLIENTE1\relatorio2.zip
CSV encontrados: 1
   OK | dados.csv -> CLIENTE1_relatorio2_dados.csv

PROCESSANDO: dados\raw\cliente\zip\CLIENTE2\relatorio1.zip
CSV encontrados: 1
   OK | dados.csv -> CLIENTE2_relatorio1_dados.csv

PROCESSANDO: dados\raw\cliente\zip\CLIENTE2\relatorio2.zip
CSV encontrados: 1
   OK | dados.csv -> CLIENTE2_relatorio2_dados.csv


RELATÓRIO FINAL
ZIP encontrados : 4
CSV extraídos   : 4
Erros           : 0
Pasta de saída  : dados\raw\cliente\csv


ARQUIVOS CSV NA PASTA DE SAÍDA
CSV | CLIENTE1_r